In [1]:
import torch
import esm
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Sử dụng GPU 0
os.environ["TORCH_HOME"] = "E:\\Dai hoc\\2526I\\dacn\\flow-matching\\.cache"

In [6]:

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
num_layers = model.num_layers
print(f"Model has {num_layers} layers.")
batch_converter = alphabet.get_batch_converter()

model.eval()  # inference mode

# Ví dụ chuỗi protein / peptide
data = [
    ("seq1", "MKTAYIAKQRQISFVKSHFSRQ"),
    ("seq2", "ACDEFGHIKLMNPQRS"),
]

batch_labels, batch_strs, batch_tokens = batch_converter(data)

with torch.no_grad():
    results = model(
        batch_tokens,
        repr_layers=[num_layers],      # layer cuối của model t6
        return_contacts=True
    )

Model has 33 layers.


In [4]:
print(len(data[0][1]))

22


In [8]:

# Lấy embedding
token_embeddings = results["representations"][num_layers]
print(token_embeddings.shape)

# Mean pooling (1 vector / sequence)
sequence_embeddings = []
for i, (_, seq) in enumerate(data):
    emb = token_embeddings[i, 1:len(seq)+1].mean(0)
    sequence_embeddings.append(emb)
sequence_embeddings = torch.stack(sequence_embeddings)
print(sequence_embeddings.shape)

torch.Size([2, 24, 1280])
torch.Size([2, 1280])


In [ ]:
print(batch_labels)

['seq1', 'seq2']


In [ ]:
print(batch_strs)


['MKTAYIAKQRQISFVKSHFSRQ', 'ACDEFGHIKLMNPQRSTVWY']


In [ ]:
print(batch_tokens)

tensor([[ 0, 20, 15, 11,  5, 19, 12,  5, 15, 16, 10, 16, 12,  8, 18,  7, 15,  8,
         21, 18,  8, 10, 16,  2],
        [ 0,  5, 23, 13,  9, 18,  6, 21, 12, 15,  4, 20, 17, 14, 16, 10,  8, 11,
          7, 22, 19,  2,  1,  1]])
